# 📞 Lead Scoring — Regresión Logística
> **Caso de negocio:** Tecsup · Admisión y Matrícula
> **Framework:** CRISP-DM · Digital Marketing Analytics — UPC
> **Autor:** Miguel Salazar · Attach Group

---

## Contexto del problema

Tecsup recibe cerca de 1,200 leads al mes desde landing pages y campañas digitales para sus carreras técnicas. El equipo de asesores de admisión llama a **todos los leads en el orden en que llegan**, sin priorizar por probabilidad real de matrícula.

**Objetivo:** Predecir qué leads tienen mayor probabilidad de matricularse para que los asesores prioricen sus llamadas.

**KPIs de éxito:**
- Tasa de matrícula +20% sobre el mismo volumen de llamadas
- Reducir el tiempo de primer contacto en leads de alta probabilidad
- AUC-ROC >= 0.75 para ordenar correctamente a los leads

**Algoritmo:** Regresión Logística (interpretable vía coeficientes y odds ratios)

## 📦 Fase 0 — Setup

In [ ]:
!pip install -q plotly

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay
)
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('Setup completo ✓')

## 1️⃣ Fase 1 — Business Understanding

In [ ]:
PROBLEMA = {
    'tipo': 'Clasificación binaria supervisada',
    'target': 'convirtio (0=no se matriculó, 1=se matriculó)',
    'ventana_prediccion': 'ciclo de admisión (≈45 días desde el registro del lead)',
    'features': [
        'dias_contacto      → días desde que el lead se registró hasta el primer contacto',
        'interacciones_web  → visitas y eventos en el sitio web (sesiones, páginas)',
        'clicks_precio      → clics en la sección de precios/pensiones',
        'tiempo_pag_min     → minutos totales navegando el sitio',
    ],
    'criterios_aceptacion': {
        'AUC-ROC': '>= 0.75',
        'Recall':  '>= 0.65  (no perder leads que sí iban a matricularse)',
        'Precision': '>= 0.55',
    }
}

for k, v in PROBLEMA.items():
    if isinstance(v, list):
        print(f'\n{k}:')
        for item in v: print(f'  {item}')
    elif isinstance(v, dict):
        print(f'\n{k}:')
        for mk, mv in v.items(): print(f'  {mk}: {mv}')
    else:
        print(f'{k}: {v}')

## 2️⃣ Fase 2 — Data Understanding

In [ ]:
# Generación de datos sintéticos con estructura realista
# En producción: reemplazar con datos de GA4 + CRM de admisión (BigQuery)
N = 1200

canales = ['Google', 'Meta', 'Organic', 'Email', 'Referido']

dias_contacto     = np.random.randint(1, 30, N)
interacciones_web = np.random.randint(0, 21, N)
clicks_precio     = np.random.randint(0, 11, N)
tiempo_pag_min    = np.random.randint(0, 31, N)
canal             = np.random.choice(canales, N)

# DGP: relaciones realistas — interés digital sube la probabilidad, la demora la baja
z = (-3.5
     - 0.06 * dias_contacto       # más días sin contactar → se enfría el lead
     + 0.18 * interacciones_web   # más visitas → más interés
     + 0.45 * clicks_precio       # mirar precios → señal fuerte de intención
     + 0.08 * tiempo_pag_min      # más tiempo navegando → más interés
     + np.random.normal(0, 1.2, N))

prob      = 1 / (1 + np.exp(-z))
convirtio = (prob > 0.5).astype(int)

df = pd.DataFrame({
    'lead_id':            range(1, N+1),
    'dias_contacto':      dias_contacto,
    'interacciones_web':  interacciones_web,
    'clicks_precio':      clicks_precio,
    'tiempo_pag_min':     tiempo_pag_min,
    'canal':              canal,
    'convirtio':          convirtio
})

print(f'Dataset: {df.shape}')
print(f'\nDistribución del target:')
print(df['convirtio'].value_counts(normalize=True).map('{:.1%}'.format))
df.head()

In [ ]:
print('=== MATRICULADOS (convirtio=1) ===')
display(df[df['convirtio']==1].describe().round(2))
print('\n=== NO MATRICULADOS (convirtio=0) ===')
display(df[df['convirtio']==0].describe().round(2))

In [ ]:
# Visualización: distribuciones por clase
features = ['dias_contacto', 'interacciones_web', 'clicks_precio', 'tiempo_pag_min']
fig = make_subplots(rows=2, cols=2, subplot_titles=features)

colors = {0: '#c0392b', 1: '#0d9488'}
for idx, feat in enumerate(features):
    r, c = idx // 2 + 1, idx % 2 + 1
    for clase, label in [(0, 'No matriculó'), (1, 'Matriculó')]:
        data = df[df['convirtio'] == clase][feat]
        fig.add_trace(
            go.Histogram(x=data, name=label, opacity=0.65,
                         marker_color=colors[clase],
                         showlegend=(idx == 0)),
            row=r, col=c
        )

fig.update_layout(barmode='overlay', height=500,
                  title='Distribución de features por clase del target',
                  plot_bgcolor='white', paper_bgcolor='white')
fig.show()

In [ ]:
# Tasa de matrícula por canal de adquisición
conv_canal = df.groupby('canal')['convirtio'].agg(['count', 'mean']).reset_index()
conv_canal.columns = ['canal', 'n_leads', 'tasa_conversion']
conv_canal = conv_canal.sort_values('tasa_conversion', ascending=False)

fig = px.bar(conv_canal, x='canal', y='tasa_conversion',
             title='Tasa de matrícula por canal de adquisición',
             text=conv_canal['tasa_conversion'].map('{:.1%}'.format),
             color='tasa_conversion', color_continuous_scale='teal')
fig.update_traces(textposition='outside')
fig.update_yaxes(tickformat='.0%')
fig.update_layout(height=380, plot_bgcolor='white', paper_bgcolor='white', showlegend=False)
fig.show()
display(conv_canal)

In [ ]:
# Matriz de correlaciones
num_cols = ['dias_contacto', 'interacciones_web', 'clicks_precio', 'tiempo_pag_min', 'convirtio']
corr = df[num_cols].corr().round(2)

fig = px.imshow(corr, text_auto=True, color_continuous_scale='RdBu_r',
                zmin=-1, zmax=1, title='Correlación entre variables')
fig.update_layout(height=400, paper_bgcolor='white')
fig.show()

## 3️⃣ Fase 3 — Data Preparation

In [ ]:
FEATURES = ['dias_contacto', 'interacciones_web', 'clicks_precio', 'tiempo_pag_min']
TARGET   = 'convirtio'

X = df[FEATURES].fillna(0)
y = df[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Escalado: la regresión logística es sensible a la escala de las variables
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'Train: {len(X_train)} | Test: {len(X_test)}')
print(f'Tasa conversión train: {y_train.mean():.1%}')
print(f'Tasa conversión test:  {y_test.mean():.1%}')

## 4️⃣ Fase 4 — Modeling

In [ ]:
# Regresión Logística: modelo lineal sobre el log-odds de conversión
model = LogisticRegression(C=1.0, max_iter=500, random_state=42)
model.fit(X_train_s, y_train)

# Validación cruzada — estimación honesta del performance
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_auc = cross_val_score(model, X_train_s, y_train, cv=cv, scoring='roc_auc')
cv_rec = cross_val_score(model, X_train_s, y_train, cv=cv, scoring='recall')

print(f'CV AUC-ROC: {cv_auc.mean():.3f} ± {cv_auc.std():.3f}')
print(f'CV Recall:  {cv_rec.mean():.3f} ± {cv_rec.std():.3f}')

In [ ]:
y_pred = model.predict(X_test_s)
y_prob = model.predict_proba(X_test_s)[:, 1]

# Coeficientes y odds ratios (variables estandarizadas)
coef_df = pd.DataFrame({
    'feature': FEATURES,
    'coef': model.coef_[0],
    'odds_ratio': np.exp(model.coef_[0])
}).sort_values('coef', ascending=True)

fig = go.Figure(go.Bar(
    x=coef_df['coef'], y=coef_df['feature'], orientation='h',
    marker_color=['#0d9488' if c > 0 else '#c0392b' for c in coef_df['coef']],
    text=coef_df['coef'].map('{:.3f}'.format), textposition='outside'
))
fig.update_layout(title='Coeficientes de la Regresión Logística (variables estandarizadas)',
                  height=350, plot_bgcolor='white', paper_bgcolor='white',
                  xaxis=dict(gridcolor='#f0f0f0', zeroline=True, zerolinecolor='#aaa'),
                  yaxis=dict(showgrid=False))
fig.show()

print('\nOdds ratios — cuánto multiplica la probabilidad de conversión por +1 desv. estándar:')
display(coef_df[['feature','coef','odds_ratio']].sort_values('odds_ratio', ascending=False))

## 5️⃣ Fase 5 — Evaluation

In [ ]:
metrics = {
    'Accuracy':  accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred, zero_division=0),
    'Recall':    recall_score(y_test, y_pred, zero_division=0),
    'F1-Score':  f1_score(y_test, y_pred, zero_division=0),
    'AUC-ROC':   roc_auc_score(y_test, y_prob),
}
criterios = {'AUC-ROC': 0.75, 'Recall': 0.65, 'Precision': 0.55}

print('=== MÉTRICAS EN TEST SET ===')
for k, v in metrics.items():
    umbral = criterios.get(k)
    estado = ''
    if umbral:
        estado = '✅ APROBADO' if v >= umbral else f'❌ NECESITA MEJORA (umbral {umbral:.2f})'
    print(f'  {k:12s}: {v:.4f}  {estado}')

In [ ]:
# Curva ROC
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)

fig = go.Figure()
fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines',
                          name=f'Logit (AUC={auc:.3f})',
                          line=dict(color='#1a4c8c', width=2.5)))
fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode='lines',
                          name='Random', line=dict(color='#ccc', dash='dash')))
fig.update_layout(title='Curva ROC — Lead Scoring',
                  xaxis_title='FPR', yaxis_title='TPR (Recall)',
                  height=400, plot_bgcolor='white', paper_bgcolor='white')
fig.show()

In [ ]:
# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(cm, display_labels=['No matriculó', 'Matriculó'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Matriz de Confusión')
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'\nVP: {tp} — leads de alta probabilidad correctamente priorizados')
print(f'FN: {fn} — leads que sí matriculan pero el modelo subestima')
print(f'FP: {fp} — llamadas "extra" a leads que no matriculan')
print(f'VN: {tn} — correctamente descartados')

In [ ]:
# Análisis de lift por decil
df_test = pd.DataFrame({'prob': y_prob, 'real': y_test.values})
df_test = df_test.sort_values('prob', ascending=False).reset_index(drop=True)

deciles = []
n = len(df_test)
for d in range(10):
    subset = df_test.iloc[d*n//10:(d+1)*n//10]
    deciles.append({'decil': f'D{d+1}', 'tasa_conversion': subset['real'].mean()})

df_lift = pd.DataFrame(deciles)
baseline = df_test['real'].mean()
df_lift['lift'] = df_lift['tasa_conversion'] / baseline

fig = px.bar(df_lift, x='decil', y='lift',
             title=f'Lift por decil (baseline = {baseline:.1%})',
             text=df_lift['lift'].map('{:.2f}x'.format),
             color='lift', color_continuous_scale='teal')
fig.add_hline(y=1, line_dash='dash', line_color='gray',
              annotation_text='Baseline (sin modelo)')
fig.update_traces(textposition='outside')
fig.update_layout(height=380, plot_bgcolor='white', paper_bgcolor='white', showlegend=False)
fig.show()
print(f'\nEl top 20% de leads por score concentra {df_lift["tasa_conversion"][:2].mean()/baseline:.1f}x más matrículas que el promedio')

## 6️⃣ Fase 6 — Deployment

In [ ]:
# Score y priorización de toda la base de leads
X_all_s = scaler.transform(df[FEATURES].fillna(0))
df['score_matricula'] = (model.predict_proba(X_all_s)[:, 1] * 100).round(1)
df['prioridad'] = pd.cut(
    df['score_matricula'],
    bins=[0, 35, 65, 100],
    labels=['Prioridad baja', 'Prioridad media', 'Prioridad alta']
)

print('Distribución de prioridad de llamada:')
print(df['prioridad'].value_counts())
print(f'\nLeads de prioridad alta: {(df["prioridad"]=="Prioridad alta").sum()}')
df[['lead_id','canal','score_matricula','prioridad']].sort_values('score_matricula', ascending=False).head(10)

In [ ]:
# Guardar outputs
df[['lead_id','canal','score_matricula','prioridad']].to_csv('leads_priorizados_tecsup.csv', index=False)
print('Archivo leads_priorizados_tecsup.csv guardado ✓')

import joblib
joblib.dump({'model': model, 'scaler': scaler, 'features': FEATURES},
            'modelo_lead_scoring_tecsup.pkl')
print('Modelo modelo_lead_scoring_tecsup.pkl guardado ✓')

In [ ]:
# Resumen ejecutivo
n_leads        = len(df)
n_alta         = (df['prioridad'] == 'Prioridad alta').sum()
tasa_conv_alta = df[df['prioridad']=='Prioridad alta']['convirtio'].mean()
tasa_conv_base = df['convirtio'].mean()
lift_conversion = tasa_conv_alta / tasa_conv_base

print('=== RESUMEN EJECUTIVO ===')
print(f'Leads del mes:                     {n_leads:,}')
print(f'Leads de prioridad alta:           {n_alta:,} ({n_alta/n_leads:.0%})')
print(f'Tasa de matrícula prioridad alta:  {tasa_conv_alta:.1%}')
print(f'Tasa de matrícula base:            {tasa_conv_base:.1%}')
print(f'Lift de conversión:                {lift_conversion:.1f}x')
print(f'\nArquitectura de producción:')
print('  GA4 + CRM admisión → BigQuery → score diario → cola priorizada de asesores')